In [ ]:
################# BCB

import requests
import pandas as pd

# Função para consultar dados da API do Banco Central do Brasil (BCB)
def consultar_indicadores_bcb(indicadores, data_inicio, data_fim):
    # Inicializando um dataframe vazio
    df_final = pd.DataFrame()

    # URL base da API do BCB
    url_base = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{}/dados?formato=json&dataInicial={}&dataFinal={}"

    # Iterando sobre cada indicador fornecido
    for indicador, nome_coluna in indicadores.items():
        # Montando a URL para o indicador
        url = url_base.format(indicador, data_inicio, data_fim)

        # Consultando a API
        response = requests.get(url)
        dados = response.json()

        # Convertendo o resultado para um DataFrame temporário
        df_temp = pd.DataFrame(dados)

        # Formatando a data e o valor
        df_temp['data'] = pd.to_datetime(df_temp['data'], format='%d/%m/%Y')
        df_temp['valor'] = pd.to_numeric(df_temp['valor'], errors='coerce')

        # Renomeando a coluna de valores para o nome do indicador
        df_temp.set_index('data', inplace=True)
        df_temp.columns = [nome_coluna]

        # Mesclando o novo indicador ao dataframe final
        if df_final.empty:
            df_final = df_temp
        else:
            df_final = df_final.join(df_temp, how='outer')

    # Retornando o dataframe final com todos os indicadores
    return df_final

# Exemplo de uso da função
indicadores_bcb = {
    "12": "CDI",     # CDI diário
    #"433": "IPCA"    # IPCA
}

data_inicio = "01/01/2024"  # Data inicial no formato dia/mês/ano
data_fim = "01/10/2024"     # Data final no formato dia/mês/ano

# Chamando a função e armazenando o resultado no dataframe
df_indicadores_bcb = consultar_indicadores_bcb(indicadores_bcb, data_inicio, data_fim)

# Exibindo as primeiras linhas do dataframe
print(df_indicadores_bcb.head())


In [ ]:
################### QUANDL


import quandl
import pandas as pd

# Função para consultar múltiplos indicadores e retornar um DataFrame com a data como index
def consultar_indicadores_quandl(indicadores, data_inicio, data_fim, api_key):
    # Configurando a API Key do Quandl
    quandl.ApiConfig.api_key = api_key
    
    # Inicializando um dataframe vazio
    df_final = pd.DataFrame()
    
    # Iterando sobre cada indicador fornecido
    for indicador in indicadores:
        # Consultando o indicador no Quandl
        df_temp = quandl.get(indicador, start_date=data_inicio, end_date=data_fim)
        
        # Renomeando a coluna de valores para o código do indicador
        df_temp.columns = [indicador]
        
        # Mesclando o novo indicador ao dataframe final
        if df_final.empty:
            df_final = df_temp
        else:
            df_final = df_final.join(df_temp, how='outer')
    
    # Retornando o dataframe final com todos os indicadores
    return df_final

# Exemplo de uso da função
indicadores = ["BCB/12", "BCB/433", "BCB/1", "BCB/7"]  # CDI, IPCA, Dólar e Ibovespa
data_inicio = "2020-01-01"
data_fim = "2024-01-01"
api_key = 'FRzYDKkncbrRPj44U7zH'

# Chamando a função e armazenando o resultado no dataframe
df_indicadores = consultar_indicadores_quandl(indicadores, data_inicio, data_fim, api_key)

# Exibindo as primeiras linhas do dataframe
print(df_indicadores.head())
